In [ ]:
import time
import pandas as pd
import requests
import numpy as np
import re

In [ ]:
import pandas as pd
from pathlib import Path

championship_audit = pd.read_csv(
    "../data/raw/bigwest_2021_2026_audit.csv",
    )


In [ ]:
api_url = (
    "https://api.parse.bot/scraper/"
    "6e80f113-decc-4a5f-960a-70b2f1d469bc/"
    "get_meet_team_scores"
)
# headers = {
#     "X-API-Key": ""
# }
new_datasets = {
    2019: 62142,
}

In [ ]:
rows = []
successful_years = []

for year, meet_id in new_datasets.items():

    try:
        response = requests.get(
            api_url,
            headers=headers,
            params={"meet_id": str(meet_id)},
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        if result.get("status") != "success":
            print(f"Failed to scrape {year}")
            continue

        data = result["data"]

        if not all(
            "events" in data.get(gender, {})
            for gender in ["men", "women"]
        ):
            print(f"Incomplete data for {year}; skipping")
            continue

        year_rows = []

        for gender in ["men", "women"]:
            for event_data in data[gender]["events"]:

                event_name = event_data["event"]
                event_lower = event_name.lower()

                if "relay" in event_lower:
                    event_type = "relay"
                    mark_unit = "seconds"

                elif any(x in event_lower for x in [
                    "heptathlon", "decathlon"
                ]):
                    event_type = "combined"
                    mark_unit = "points"

                elif any(x in event_lower for x in [
                    "jump", "vault", "shot put",
                    "discus", "hammer", "javelin"
                ]):
                    event_type = "field"
                    mark_unit = "meters"

                else:
                    event_type = "track"
                    mark_unit = "seconds"

                for entry in event_data["results"]:

                    athletes = entry.get("athletes") or []
                    
                    athlete = (
                        athletes[0]
                        if athletes and event_type != "relay"
                        else {}
                    )

                    year_rows.append({
                        "athlete_id": athlete.get("athlete_id"),
                        "athlete_name": athlete.get("name"),
                        "team": entry.get("team"),
                        "meet_name": data.get("meet_name"),
                        "meet_year": year,
                        "gender": gender,
                        "event": event_name,
                        "mark": entry.get("mark"),
                        "mark_unit": mark_unit,
                        "place": entry.get("place"),
                        "wind": entry.get("wind"),
                        "event_type": event_type,
                        "points": entry.get("points")
                    })

        if not year_rows:
            print(f"No event results for {year}; skipping")
            continue

        rows.extend(year_rows)
        successful_years.append(year)

        print(f"Scraped {year}: {len(year_rows)} rows")

    except (requests.RequestException, ValueError) as e:
        print(f"Error scraping {year}: {e}")

    time.sleep(3.25)


In [ ]:
new_results = pd.DataFrame(rows)
new_results["athlete_id"] = (
    new_results["athlete_id"].astype("string")
)

championship_audit = pd.concat(
    [championship_audit, new_results],
    ignore_index=True
)

championship_audit = (
    championship_audit
    .drop_duplicates()
    .sort_values(
        ["meet_year", "gender", "event"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)
championship_audit.to_csv(
    "../data/raw/bigwest_2021_2026_audit.csv",
    index=False
)

Cleaning Big west

In [ ]:
TEAM_NAMES = {
    "CSUN":              ["CSUN", "Cal St. Northridge", "Cal State Northridge"],
    "Cal St. Fullerton": ["Cal St. Fullerton", "Cal State Fullerton", "CSUF"],
    "Long Beach St.":    ["Long Beach St.", "Long Beach State", "LBSU"],
    "Hawaii":            ["Hawaii", "Hawai'i", "Hawaiʻi"],
    "Cal Poly":          ["Cal Poly"],
    "CSU Bakersfield":   ["CSU Bakersfield", "Cal St. Bakersfield", "Cal State Bakersfield"],
    "UC Davis":          ["UC Davis"],
    "UC Irvine":         ["UC Irvine"],
    "UC Riverside":      ["UC Riverside"],
    "UC San Diego":      ["UC San Diego", "UCSD"],
    "UC Santa Barbara":  ["UC Santa Barbara", "UCSB"],
}
_lookup = {re.sub(r"\s+", " ", v).strip().upper(): std for std, vs in TEAM_NAMES.items() for v in vs}

def clean_team(s):
    key = s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.upper()
    out = key.map(_lookup)
    unknown = sorted(key[out.isna() & s.notna()].unique())
    if unknown:
        print("Unrecognized team names (add them to TEAM_NAMES):", unknown)
    return out.fillna(s)

df = championship_audit.copy()
df["team"] = clean_team(df["team"])

df["is_prelim"] = df["event"].str.contains(
    r"\b(?:Preliminaries|Prelims)\b",
    case=False,
    na=False
)

df["athlete_id"] = (
    pd.to_numeric(df["athlete_id"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["is_final"] = df["event"].str.contains(
    r"\bFinals\b",
    case=False,
    na=False
)

# Standardize event names
df["event"] = df["event"].str.replace(
    r"\s+(?:Preliminaries|Prelims|Finals)$",
    "",
    regex=True,
    case=False
)

df["original_place"] = pd.to_numeric(
    df["place"], errors="coerce"
)

df["points"] = pd.to_numeric(
    df["points"], errors="coerce"
).fillna(0)

# Keep track of suspected DNS athletes
df["suspected_dns_final"] = False

keys = ["meet_year", "gender", "event"]

processed_groups = []

for group_key, group in df.groupby(keys, sort=False):

    group = group.copy()

    prelims = group[group["is_prelim"]].copy()
    finals = group[group["is_final"]].copy()

    # Events without preliminaries
    if prelims.empty:
        processed_groups.append(group)
        continue

    # Use the top 8 preliminary places as presumed qualifiers
    qualifiers = prelims[
        prelims["original_place"].between(1, 8)
    ].copy()

    # Athletes appearing in finals
    finalist_ids = set(
        finals["athlete_id"].dropna()
    )

    qualifier_ids = set(
        qualifiers["athlete_id"].dropna()
    )

    # Athletes who qualified but have no finals result
    missing_finalists = qualifier_ids - finalist_ids

    # Remove prelim results for athletes with finals results
    prelims = prelims[
        ~prelims["athlete_id"].isin(finalist_ids)
    ].copy()

    # Flag missing finalists
    dns_mask = prelims["athlete_id"].isin(missing_finalists)

    prelims.loc[dns_mask, "suspected_dns_final"] = True
    prelims.loc[dns_mask, "mark"] = "DNS"
    prelims.loc[dns_mask, "points"] = 0

    # Keep original prelim mark for non-qualifiers
    prelims.loc[~dns_mask, "points"] = 0

    # Keep any non-prelim/non-final rows too
    other = group[
        ~group["is_prelim"] & ~group["is_final"]
    ]

    combined = pd.concat(
        [finals, prelims, other],
        ignore_index=True
    )

    # Sort: finalists, suspected DNS, then non-qualifiers
    combined["sort_group"] = np.select(
        [
            combined["is_final"],
            combined["suspected_dns_final"]
        ],
        [0, 1],
        default=2
    )

    combined = combined.sort_values(
        ["sort_group", "original_place"],
        na_position="last"
    )

    # Rank actual finalists using their finals places
    actual_final_mask = combined["is_final"]

    combined.loc[actual_final_mask, "place"] = (
        combined.loc[actual_final_mask, "original_place"]
    )

    # Missing finalists have no official final placement
    combined.loc[
        combined["suspected_dns_final"], "place"
    ] = pd.NA

    # Non-qualifiers start at 9
    nonqual_mask = (
        ~combined["is_final"]
        & ~combined["suspected_dns_final"]
    )

    combined.loc[nonqual_mask, "place"] = (
        np.arange(nonqual_mask.sum()) + 9
    )

    processed_groups.append(combined)

# Combine all events
championship_processed = pd.concat(
    processed_groups,
    ignore_index=True
)

# Show missing-finalist status separately
suspected_dns = championship_processed[
    championship_processed["suspected_dns_final"]
].copy()


# Remove temporary columns, but keep DNS flag
championship_processed = championship_processed.drop(
    columns=[
        "is_prelim",
        "is_final",
        "original_place",
        "sort_group",
        "suspected_dns_final"
    ],
    errors="ignore"
)

championship_processed.to_csv(
    "../data/processed/bigwest_2019_2026_results.csv",
    index=False
)